In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [9]:
DATA_PATH = "usina_with_outliers.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Shape: (9568, 5)

Columns:
['AT', 'V', 'AP', 'RH', 'PE']


,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90


In [10]:
# Q1: Cook's Distance outlier detection with Statsmodels

# Define predictors (X) and target (y)
X = df.drop(columns=["PE"])
y = df["PE"]

# Add intercept column for OLS
X = sm.add_constant(X)

# Fit OLS regression
model = sm.OLS(y, X).fit()

# Compute influence measures
influence = model.get_influence()
cooks_d = influence.cooks_distance[0]  # array of Cook's Distance values

# Identify outliers using threshold
# Standard rule: 4/n
n = len(df)
threshold = 4 / n
outliers = np.where(cooks_d > threshold)[0]

print(f"Detected {len(outliers)} outliers at indices: {outliers}")

# 6️⃣ Remove outliers
df_clean = df.drop(index=outliers).reset_index(drop=True)

# 7️⃣ Export cleaned dataset
df_clean.to_csv("usina.csv", index=False)
print("Cleaned dataset exported to 'usina.csv'.")

Detected 120 outliers at indices: [  35   49  112  339  402  418  511  526  606  875  927 1091 1130 1247
 1290 1348 1475 1524 1662 1735 1779 1835 1907 1914 1985 2047 2048 2134
 2221 2338 2360 2420 2538 2556 2643 2776 2846 2876 2972 3117 3243 3383
 3452 3527 3617 3895 4193 4206 4218 4228 4234 4435 4529 4542 4719 4737
 4741 4794 4842 4872 4900 4903 5099 5177 5260 5480 5535 5830 5916 5922
 5996 6116 6338 6482 6603 6658 6768 6832 6881 7091 7094 7136 7182 7351
 7398 7536 7564 7625 7664 7678 7702 7750 7791 7846 7900 7902 7926 7941
 7944 8098 8160 8284 8415 8501 8656 8717 8738 8951 8953 8983 9153 9219
 9241 9245 9406 9408 9463 9472 9510 9525]
Cleaned dataset exported to 'usina.csv'.


In [14]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

target_col = "PE"

# Prepare features and target
def prepare_data(df):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y

X_out, y_out = prepare_data(df)
X_clean, y_clean = prepare_data(df_clean)

# Split into train/test
def split_data(X, y, test_size=0.2, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

X_out_tr, X_out_te, y_out_tr, y_out_te = split_data(X_out, y_out)
X_clean_tr, X_clean_te, y_clean_tr, y_clean_te = split_data(X_clean, y_clean)

# Define metrics function
def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    metrics = {
        "MAE_train": mean_absolute_error(y_tr, y_tr_pred),
        "MAE_test": mean_absolute_error(y_te, y_te_pred),
        "MSE_train": mean_squared_error(y_tr, y_tr_pred),
        "MSE_test": mean_squared_error(y_te, y_te_pred),
        "R2_train": r2_score(y_tr, y_tr_pred),
        "R2_test": r2_score(y_te, y_te_pred),
    }
    return metrics

# Train models on a dataset
def train_models(X_tr, y_tr, X_te, y_te):
    results = {}

    # 5a. Linear Regression
    lr = LinearRegression()
    lr.fit(X_tr, y_tr)
    results["LinearRegression"] = evaluate_model(lr, X_tr, y_tr, X_te, y_te)

    # 5b. Ridge Regression
    ridge_alphas = [0.01, 0.1, 1, 10, 100]
    for alpha in ridge_alphas:
        ridge = Ridge(alpha=alpha)
        ridge.fit(X_tr, y_tr)
        results[f"Ridge_alpha_{alpha}"] = evaluate_model(ridge, X_tr, y_tr, X_te, y_te)

    # 5c. Lasso Regression
    lasso_alphas = [0.01, 0.1, 1, 10, 100]
    for alpha in lasso_alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_tr, y_tr)
        results[f"Lasso_alpha_{alpha}"] = evaluate_model(lasso, X_tr, y_tr, X_te, y_te)

    return results

# Evaluate on dataset with outliers
print("==== With Outliers ====")
results_out = train_models(X_out_tr, y_out_tr, X_out_te, y_out_te)
for model_name, metrics in results_out.items():
    print(f"\n{model_name}:")
    print(metrics)

# Evaluate on cleaned dataset
print("\n==== After Removing Outliers ====")
results_clean = train_models(X_clean_tr, y_clean_tr, X_clean_te, y_clean_te)
for model_name, metrics in results_clean.items():
    print(f"\n{model_name}:")
    print(metrics)

==== With Outliers ====

LinearRegression:
{'MAE_train': 5.141647701003086, 'MAE_test': 5.202587765711338, 'MSE_train': 117.97505618149496, 'MSE_test': 146.4095626666161, 'R2_train': 0.6621072626441313, 'R2_test': 0.5965585328076742}

Ridge_alpha_0.01:
{'MAE_train': 5.141647753335167, 'MAE_test': 5.202587810302384, 'MSE_train': 117.975056181495, 'MSE_test': 146.40956249006018, 'R2_train': 0.6621072626441311, 'R2_test': 0.5965585332941861}

Ridge_alpha_0.1:
{'MAE_train': 5.141648224323733, 'MAE_test': 5.20258821162161, 'MSE_train': 117.97505618150107, 'MSE_test': 146.40956090106084, 'R2_train': 0.6621072626441138, 'R2_test': 0.5965585376727813}

Ridge_alpha_1:
{'MAE_train': 5.1416529341877295, 'MAE_test': 5.202592224794984, 'MSE_train': 117.97505618210734, 'MSE_test': 146.40954501152396, 'R2_train': 0.6621072626423774, 'R2_test': 0.5965585814574765}

Ridge_alpha_10:
{'MAE_train': 5.1417000306574945, 'MAE_test': 5.202632354631402, 'MSE_train': 117.97505624272918, 'MSE_test': 146.40938616

Results Discussion:
1. Outliers significantly affect training and testing metrics. Most error metrics have been boosted by about 400% due to these outliers.
2. The dataset without outliers shows better generalization by far.
3. Lasso and Ridge models do not have a significant effect on the traditional linear regression model. They don't help in this scenario.